# Session 2: tie lines and the lever rule
**A. Real-world problem.** A foundry wants a casting that is exactly half liquid and half solid
(a "semi-solid" slurry that can be pressed into shape). Which alloy composition and which
temperature give that? Session 1 told us only the region; today we get numbers: how much
of each phase there is, and what each phase is made of.

**Illustrative diagram.** As in Session 1 the liquidus and solidus are straight-line pieces
between Cu 1085 C and Ni 1455 C. They are for teaching, not the measured Cu-Ni diagram.
Composition is in weight percent (wt%) Ni.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

X_NODES = np.array([0.0, 50.0, 100.0])
LIQ_NODES = np.array([1085.0, 1320.0, 1455.0])    # liquidus temperatures (C) at those compositions
SOL_NODES = np.array([1085.0, 1240.0, 1455.0])    # solidus temperatures (C)


def liquidus(x):
    return np.interp(x, X_NODES, LIQ_NODES)


def solidus(x):
    return np.interp(x, X_NODES, SOL_NODES)

## Step 1: read the phase compositions off a tie line
At temperature T draw a horizontal line (a *tie line*) through the two-phase band.
Where it meets the liquidus the liquid has composition C_L; where it meets the solidus the
solid has composition C_S. We find them by reading the boundary lines *sideways*:
the inverse of "temperature at composition" is "composition at temperature".

In [ ]:
def liquid_composition(T):
    """C_L: composition (wt% Ni) of the liquid on the liquidus at temperature T."""
    return float(np.interp(T, LIQ_NODES, X_NODES))


def solid_composition(T):
    """C_S: composition (wt% Ni) of the solid on the solidus at temperature T."""
    return float(np.interp(T, SOL_NODES, X_NODES))

## Step 2: the lever rule (derived)
Let the alloy have overall composition C0 and let a fraction f_L be liquid, f_S = 1 - f_L solid.
*Nickel is conserved.* The Ni carried by the liquid plus the Ni carried by the solid is all the Ni:

    C0 = f_L * C_L + (1 - f_L) * C_S

Solve for f_L:   C0 - C_S = f_L (C_L - C_S)   so

    f_L = (C_S - C0) / (C_S - C_L)      f_S = (C0 - C_L) / (C_S - C_L)

In words: the fraction of a phase is the *opposite arm* of the tie line divided by the *whole*
tie line. The liquid fraction uses the arm from C0 to C_S (the far end).

In [ ]:
def tie_line(C0, T):
    """Return (C_L, C_S, f_L, f_S) for alloy C0 at temperature T (C)."""
    tl, ts = float(liquidus(C0)), float(solidus(C0))
    if T >= tl:
        return C0, float("nan"), 1.0, 0.0       # fully liquid: no solid yet
    if T <= ts:
        return float("nan"), C0, 0.0, 1.0       # fully solid
    cl, cs = liquid_composition(T), solid_composition(T)
    fl = (cs - C0) / (cs - cl)
    return cl, cs, fl, 1.0 - fl


def liquid_fraction(C0, T):
    return tie_line(C0, T)[2]

## Step 3: the inverse (design) problem
* Fixed composition, find the temperature for a wanted liquid fraction: f_L falls from 1 at the
  liquidus to 0 at the solidus, so bisection finds T.
* Fixed temperature, find the composition: a 50 percent liquid alloy sits exactly in the
  middle of the tie line, C0 = (C_L + C_S) / 2. More generally C0 = f_L C_L + (1 - f_L) C_S.

In [ ]:
def temperature_for_liquid_fraction(C0, target):
    """Temperature (C) where alloy C0 has liquid fraction `target` (0 < target < 1)."""
    lo, hi = float(solidus(C0)), float(liquidus(C0))
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        if liquid_fraction(C0, mid) > target:      # too liquid: go colder
            hi = mid
        else:
            lo = mid
    return 0.5 * (lo + hi)


def composition_for_liquid_fraction(T, target):
    """Overall composition C0 giving liquid fraction `target` at temperature T (on the tie line)."""
    return target * liquid_composition(T) + (1.0 - target) * solid_composition(T)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    C0, T = 50.0, 1280.0
    cl, cs, fl, fs = tie_line(C0, T)
    print(f"alloy {C0:.0f} wt% Ni at {T:.0f} C: C_L = {cl:.1f}, C_S = {cs:.1f}, f_L = {fl:.3f}, f_S = {fs:.3f}")
    print(f"50% liquid for C0 = 50: T = {temperature_for_liquid_fraction(50.0, 0.5):.1f} C")
    for Tq in (1250, 1300, 1350):
        print(f"50% liquid at {Tq} C needs C0 = {composition_for_liquid_fraction(Tq, 0.5):.1f} wt% Ni")

    xs = np.linspace(0, 100, 201)
    fig, ax = plt.subplots(figsize=(6.4, 4.8))
    ax.plot(xs, liquidus(xs), "r-", label="liquidus")
    ax.plot(xs, solidus(xs), "b-", label="solidus")
    ax.plot([cl, cs], [T, T], "k-", lw=2, label="tie line")
    ax.plot([C0], [T], "ko")
    ax.annotate("C_L", (cl, T), xytext=(cl - 6, T + 18))
    ax.annotate("C0", (C0, T), xytext=(C0 - 2, T - 32))
    ax.annotate("C_S", (cs, T), xytext=(cs + 1, T + 18))
    ax.set_xlabel("composition (weight percent Ni)")
    ax.set_ylabel("temperature (C)")
    ax.set_title(f"Tie line at {T:.0f} C: f_L = {fl:.2f}, f_S = {fs:.2f}")
    ax.legend(loc="lower right")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "tie_line.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.2, 4.4))
    for c in (30.0, 50.0, 70.0):
        Ts = np.linspace(float(solidus(c)), float(liquidus(c)), 100)
        ax2.plot(Ts, [liquid_fraction(c, t) for t in Ts], label=f"C0 = {c:.0f} wt% Ni")
    ax2.set_xlabel("temperature (C)")
    ax2.set_ylabel("liquid fraction f_L")
    ax2.set_title("Liquid fraction falls from 1 to 0 across the band")
    ax2.legend()
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "liquid_fraction.png", dpi=120)

## Where this stops
The lever rule assumes the phases always reach equilibrium (every atom has time to move).
Session 3 simulates cooling, adds a eutectic, and shows when this assumption breaks.

## Coding tasks
1. **Easy.** Use `tie_line(50, 1300)` and print the four values; compute the fractions by hand and compare with your answer to question B in Section F.
2. **Stretch.** Find, with `temperature_for_liquid_fraction`, the temperature at which a 30 wt% Ni alloy is 25 percent liquid and check it with `liquid_fraction`.
3. **Explain.** Explain, with a sketch of a seesaw, why an alloy at the middle of the tie line is half liquid.

## Assessment
- **A. Conceptual.** Why is the liquid fraction given by the arm to the far end of the tie line (C_S - C0) and not the near arm?
- **B. Numerical.** For a 30 wt% Ni alloy at 1200 C, find C_L, C_S and the liquid fraction.
- **C. Graph and simulation interpretation.** On the liquid-fraction graph, which alloy of 30, 50 and 70 wt% Ni has the widest temperature band? What does the steepness of a curve tell you?
- **D. Design and optimisation challenge.** Design: at 1300 C which alloy composition is exactly half liquid? Which temperature makes a 50 wt% Ni alloy half liquid? Give a second pair that works.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    cl, cs, fl, fs = L.tie_line(50.0, 1300.0)
    # hand calculation: C_L = 50*(1300-1085)/(1320-1085)*... read sideways off the straight pieces
    cl_hand = (1300.0 - 1085.0) / (1320.0 - 1085.0) * 50.0          # liquidus piece 0..50
    cs_hand = 50.0 + (1300.0 - 1240.0) / (1455.0 - 1240.0) * 50.0   # solidus piece 50..100
    checks.append((f"tie line at 1300 C: C_L = {cl_hand:.1f} and C_S = {cs_hand:.1f} wt% Ni",
                   abs(cl - cl_hand) < 1e-9 and abs(cs - cs_hand) < 1e-9 and round(cl, 1) == 45.7 and round(cs, 1) == 64.0))
    fl_hand = (cs_hand - 50.0) / (cs_hand - cl_hand)
    checks.append((f"worked example: 50 wt% Ni at 1300 C is {100 * fl_hand:.1f} percent liquid (lever rule)",
                   abs(fl - fl_hand) < 1e-12 and round(100 * fl, 1) == 76.6))
    checks.append(("fractions add to 1", abs(fl + fs - 1.0) < 1e-12))
    checks.append(("mass balance: f_L C_L + f_S C_S = C0 for many (C0, T)",
                   all(abs(f * a + (1 - f) * b - c0) < 1e-9
                       for c0 in (20.0, 35.0, 50.0, 80.0)
                       for T in np.linspace(L.solidus(c0) + 1, L.liquidus(c0) - 1, 7)
                       for a, b, f, _ in [L.tie_line(c0, float(T))])))
    checks.append(("edge: at the liquidus f_L = 1 and at the solidus f_L = 0",
                   abs(L.liquid_fraction(50.0, 1320.0) - 1.0) < 1e-12 and abs(L.liquid_fraction(50.0, 1240.0)) < 1e-12))
    checks.append(("edge: above the liquidus the alloy is all liquid, below the solidus all solid",
                   L.liquid_fraction(50.0, 1400.0) == 1.0 and L.liquid_fraction(50.0, 1100.0) == 0.0))
    checks.append(("liquid fraction never increases as the alloy cools",
                   bool(np.all(np.diff([L.liquid_fraction(60.0, float(t)) for t in np.linspace(1400, 1100, 61)]) <= 1e-12))))
    t50 = L.temperature_for_liquid_fraction(50.0, 0.5)
    checks.append((f"design: 50 wt% Ni is half liquid at {t50:.1f} C (check f_L = 0.5)",
                   abs(L.liquid_fraction(50.0, t50) - 0.5) < 1e-9 and abs(t50 - 1278.2) < 0.05))
    c1300 = L.composition_for_liquid_fraction(1300.0, 0.5)
    checks.append((f"design: at 1300 C the alloy that is half liquid is {c1300:.1f} wt% Ni (middle of the tie line)",
                   abs(c1300 - 54.85) < 0.01 and abs(L.liquid_fraction(c1300, 1300.0) - 0.5) < 1e-9))
    t70 = L.temperature_for_liquid_fraction(70.0, 0.5)
    checks.append((f"design: 70 wt% Ni is half liquid at {t70:.1f} C",
                   abs(t70 - 1355.5) < 0.05))
    checks.append(("pure metals have no two-phase band: liquidus = solidus at 0 and 100 wt% Ni",
                   L.liquidus(0.0) == L.solidus(0.0) and L.liquidus(100.0) == L.solidus(100.0)))
    cl2, cs2, fl2, fs2 = L.tie_line(50.0, 1280.0)
    checks.append((f"50 wt% Ni at 1280 C: C_L = {cl2:.1f}, C_S = {cs2:.1f} wt% Ni, f_L = {fl2:.3f}",
                   (round(cl2, 1), round(cs2, 1), round(fl2, 3)) == (41.5, 59.3, 0.522)))
    cl3, cs3, fl3, _ = L.tie_line(30.0, 1200.0)
    checks.append((f"30 wt% Ni at 1200 C: C_L = {cl3:.1f}, C_S = {cs3:.1f}, f_L = {fl3:.3f}",
                   (round(cl3, 1), round(cs3, 1), round(fl3, 3)) == (24.5, 37.1, 0.562)))
    checks.append(("half liquid needs 43.7 wt% Ni at 1250 C and 68.3 wt% Ni at 1350 C",
                   round(L.composition_for_liquid_fraction(1250.0, 0.5), 1) == 43.7 and round(L.composition_for_liquid_fraction(1350.0, 0.5), 1) == 68.3))
    f_app = L.liquid_fraction(70.0, 1355.0)
    checks.append((f"app challenge: 70 wt% Ni at 1355 C is {100 * f_app:.1f} percent liquid, within 3 points of half",
                   abs(f_app - 0.5) < 0.03 and abs(L.liquid_fraction(50.0, 1280.0) - 0.5) >= 0.02))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
